# 02 — Oracle Retrain (Θ_r) — Retain-Only Gold-Standard Baseline

**Pipeline role:** Second notebook in the CMF unlearning pipeline. Consumes
artifacts produced by NB1 and produces:
1. One Oracle checkpoint per `(seed, forget_class)` pair
   (`oracle_dir/<experiment_name>_oracle_seed<N>_fc<C><suffix>.pt`)
2. Per-run training logs (`<name>_oracle_seed<N>_fc<C><suffix>_trainlog.json`)
3. `oracle_summary.csv` — one row per experiment with the full three-metric
   evaluation (Output, Linear Probe, NCC) used by downstream NB3–NB4.

**Oracle definition:** A fresh model trained **from scratch** on the retain
training subset only (the gold-standard / retrain baseline of the paper).
No fine-tuning of any existing checkpoint — every Oracle model is randomly
initialised and trained with the same schedule as NB1 (cosine + warmup +
early stopping), using `RETRAIN_CFG` hyperparameters (200 epochs, lr=0.01
for CIFAR, Table 4).

**Inputs consumed from NB1:**
- `pretrain_dir/<experiment_name>_seed<N><suffix>.pt` — metadata cross-reference
  only; the Oracle model is never fine-tuned from it.
- `splits_dir/<experiment_name>_seed<N>_fc<C><suffix>.json` — provides
  `retain_train_idx`, `retain_test_idx`, and `forget_test_idx`; **never
  regenerated here**.

## How to use
1. **Cell 0** — clones the repo and installs dependencies (same as NB1; skip if already done).
2. **Cell 1** — set `DATASET = "cifar10"` or `DATASET = "cifar100"`.  That is the only edit needed.
3. Run all remaining cells.

---
## Cell 0 — Setup: clone repo & install dependencies

Identical to NB1 Cell 0.  Skip if the environment is already prepared.

In [ ]:
import subprocess, sys, os

REPO_URL  = "https://github.com/tiensinh2/CMF_UNLearning_Posthoc.git"
REPO_NAME = "CMF_UNLearning_Posthoc"   # folder cloned into

# ── clone or pull to always use the latest code ──────────────────────────────
if not os.path.isdir(REPO_NAME):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    print("Clone complete.")
else:
    print(f"Repo exists — pulling latest changes from origin ...")
    subprocess.run(["git", "-C", REPO_NAME, "pull", "--ff-only"], check=True)
    print("Pull complete.")

# ── install dependencies ─────────────────────────────────────────────────────
_req = os.path.join(REPO_NAME, "requirements.txt")
if os.path.isfile(_req):
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", _req], check=True)
    print("Dependencies installed.")
else:
    # Minimal hard-coded fallback if requirements.txt is absent
    _pkgs = ["torch", "torchvision", "pytorch-lightning", "torchmetrics",
             "pyyaml", "pandas", "numpy", "timm"]
    print(f"requirements.txt not found — installing: {_pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _pkgs, check=True)

# ── add repo root to sys.path ────────────────────────────────────────────────
_NB_DIR = os.path.abspath(REPO_NAME)
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)

print(f"Repo root on sys.path: {_NB_DIR}")

---
## Cell 1 — Choose dataset  ✏️  ← the ONLY edit needed

Set `DATASET` to `"cifar10"` or `"cifar100"`.  
The notebook will automatically load the matching config file from `configs/`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  DATASET — the ONLY value you should change between experiments.        ║
# ║  "cifar10"  → loads configs/nb2_config_cifar10.yaml                    ║
# ║  "cifar100" → loads configs/nb2_config_cifar100.yaml                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

DATASET = "cifar10"   # ← change to "cifar100" for CIFAR-100

---
## Cell 2 — Imports

In [ ]:
import copy
import json
import random
import time
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional

import yaml

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR

# ── project modules ──────────────────────────────────────────────────────────
from paper_hparams import GLOBAL_CFG, PRETRAIN_CFG, RETRAIN_CFG, CKPT_DIRS
from train import train
from unlearn.cmf_weights import ModelModule
from utils import (
    get_dataset,
    test,
    to_jsonable,
)

# ── shared evaluation pipeline (used identically in NB3–NB4) ─────────────────
from nb_eval_helpers import (
    eval_cmf_three_metrics,
    build_test_split_loaders,
)

print("All imports OK.")

---
## Cell 3 — Load & resolve configuration

Selects `configs/nb2_config_{DATASET}.yaml` based on the `DATASET` variable
set in Cell 1, loads it, and merges with `paper_hparams.RETRAIN_CFG` defaults
(Oracle uses `RETRAIN_CFG`, not `PRETRAIN_CFG`).

In [ ]:
# ── select config file from DATASET variable ─────────────────────────────────
_CONFIG_MAP = {
    "cifar10":  "configs/nb2_config_cifar10.yaml",
    "cifar100": "configs/nb2_config_cifar100.yaml",
}
assert DATASET in _CONFIG_MAP, (
    f"Unknown DATASET '{DATASET}'. Choose from: {list(_CONFIG_MAP.keys())}"
)
_config_path = Path(_NB_DIR) / _CONFIG_MAP[DATASET]
assert _config_path.exists(), f"Config file not found: {_config_path}"
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg: Dict[str, Any] = yaml.safe_load(_f)
print(f"DATASET='{DATASET}' → loaded config: {_config_path}")

# ── merge: RETRAIN_CFG defaults first, then YAML overrides ───────────────────
_hp = {**RETRAIN_CFG}                       # Oracle uses RETRAIN_CFG defaults
_hp_overrides = _cfg.get("hparam_overrides") or {}
_hp.update(_hp_overrides)                    # intentional ablation overrides

# ── top-level protocol fields ────────────────────────────────────────────────
EXPERIMENT_NAME  = _cfg["experiment_name"]
SUFFIX           = _cfg.get("suffix", "")
DATASET          = _cfg["dataset"]
ARCH             = _cfg["arch"]
DATA_PATH        = _cfg.get("data_path",        GLOBAL_CFG["data_path"])

# ── Kaggle: redirect data_path to writable /kaggle/working/data ─────────────
# /kaggle/input/ is read-only; torchvision's download=True would crash there.
# If the Kaggle CIFAR dataset is attached, symlink its already-extracted
# folder into the writable path so no re-download is needed.
# If only the .tar.gz is present, extract it into the writable path.
_KAGGLE_INPUT = Path("/kaggle/input")
if _KAGGLE_INPUT.exists():
    DATA_PATH = "/kaggle/working/data"
    _writable_data = Path(DATA_PATH)
    _writable_data.mkdir(parents=True, exist_ok=True)
    _CIFAR_FOLDER_MAP = {"cifar10": "cifar-10-batches-py", "cifar100": "cifar-100-python"}
    _expected_folder = _CIFAR_FOLDER_MAP.get(DATASET)
    if _expected_folder and not (_writable_data / _expected_folder).exists():
        import glob as _glob
        # 1) Try to symlink the pre-extracted folder directly
        _matches = _glob.glob(f"/kaggle/input/**/{_expected_folder}", recursive=True)
        if _matches:
            (_writable_data / _expected_folder).symlink_to(_matches[0])
            print(f"[Kaggle] Symlinked {_matches[0]} -> {_writable_data / _expected_folder}")
        else:
            # 2) Fall back: find the .tar.gz and extract it into the writable dir
            _TARBALL_MAP = {"cifar10": "cifar-10-python.tar.gz", "cifar100": "cifar-100-python.tar.gz"}
            _tarball_name = _TARBALL_MAP.get(DATASET)
            _tar_matches = _glob.glob(f"/kaggle/input/**/{_tarball_name}", recursive=True) if _tarball_name else []
            if _tar_matches:
                import tarfile as _tarfile
                print(f"[Kaggle] Extracting {_tar_matches[0]} -> {DATA_PATH} ...")
                with _tarfile.open(_tar_matches[0], 'r:gz') as _tf:
                    _tf.extractall(DATA_PATH)
                print(f"[Kaggle] Extraction complete.")
            else:
                print(f"[Kaggle] WARNING: neither '{_expected_folder}' folder nor '{_tarball_name}' "
                      "found in /kaggle/input. torchvision will attempt to download.")
    print(f"[Kaggle] data_path overridden to: {DATA_PATH}")

SEEDS: List[int] = _cfg.get("SEEDS",             GLOBAL_CFG["SEEDS"])
TEST_MODE: bool  = bool(_cfg.get("TEST_MODE",    GLOBAL_CFG["TEST_MODE"]))
TEST_EPOCHS_SCALE: float = float(
    _cfg.get("TEST_EPOCHS_SCALE", GLOBAL_CFG["TEST_EPOCHS_SCALE"])
)

BATCH_SIZE      = int(_cfg.get("batch_size",      GLOBAL_CFG["batch_size"]))
TEST_BATCH_SIZE = int(_cfg.get("test_batch_size", GLOBAL_CFG["test_batch_size"]))
NUM_WORKERS     = int(_cfg.get("num_workers",     GLOBAL_CFG["num_workers"]))

# ── output directories ───────────────────────────────────────────────────────
PRETRAIN_DIR = Path(_cfg.get("pretrain_dir", CKPT_DIRS["pretrain"]))
SPLITS_DIR   = Path(_cfg.get("splits_dir",   CKPT_DIRS["splits"]))
ORACLE_DIR   = Path(_cfg.get("oracle_dir",   CKPT_DIRS["oracle"]))
RESULTS_DIR  = Path(_cfg.get("results_dir",  CKPT_DIRS["results"]))

ORACLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── test-mode epoch scaling ──────────────────────────────────────────────────
if TEST_MODE:
    _hp["epochs"] = max(1, int(_hp["epochs"] * TEST_EPOCHS_SCALE))
    SUFFIX = (SUFFIX or "") + "_test"
    print(f"[TEST_MODE] epochs reduced to {_hp['epochs']}, suffix set to '{SUFFIX}'")

# ── forget_classes resolved after dataset load ───────────────────────────────
_forget_classes_cfg = _cfg.get("forget_classes")   # None → all classes

print("=" * 60)
print(f"Experiment : {EXPERIMENT_NAME}{SUFFIX}")
print(f"Dataset    : {DATASET}")
print(f"Arch       : {ARCH}")
print(f"Seeds      : {SEEDS}")
print(f"Test mode  : {TEST_MODE}")
print(f"Oracle HP  : {_hp}")
print(f"HP overrides applied: {_hp_overrides}")
print("=" * 60)

---
## Cell 4 — Build `args` namespace

Constructs the `SimpleNamespace` that the project's `utils.py`,
`unlearn/cmf_weights.py`, and `nb_eval_helpers.py` expect.  All values come
from the YAML config / merged hyperparameter dict — never hardcoded here.

`unlearn_method = "pre_train"` keeps the Oracle on the same code path as NB1
full training (no unlearning logic applied).

In [ ]:
import types

args = types.SimpleNamespace(
    # ── dataset / arch ─────────────────────────────────────────────
    dataset          = DATASET,
    arch             = ARCH,
    data_path        = DATA_PATH,
    train_transform  = True,       # always use augmentation for Oracle training
    num_classes      = -1,         # filled in by get_dataset
    class_label_names= [],         # filled in by get_dataset

    # ── unlearn_method: "pre_train" keeps Oracle on the standard train path ──
    unlearn_method   = "pre_train",
    unlearn_class    = [],

    # ── CMF model flags (required by ModelModule and nb_eval_helpers) ────────
    CMFClassifier    = True,
    remove_FC        = True,
    CMF_momentum     = float(_hp.get("CMF_momentum", PRETRAIN_CFG.get("CMF_momentum", 0.9))),
    temperature      = float(_hp.get("temperature",  PRETRAIN_CFG.get("temperature",  1.0))),
    pretrained       = False,

    # ── optimiser / training ────────────────────────────────────────
    lr               = float(_hp["lr"]),
    momentum         = float(_hp["momentum"]),
    weight_decay     = float(_hp["weight_decay"]),
    nesterov         = bool(_hp["nesterov"]),
    epochs_or_steps  = int(_hp["epochs"]),

    # ── scheduler ───────────────────────────────────────────────────
    lr_scheduler     = _hp.get("scheduler", "cosine"),
    warmup_epochs    = int(_hp.get("warmup_epochs", 5)),
    min_lr           = 1e-5,
    patience         = int(_hp.get("early_stop_patience", 50)),

    # ── dataloader ──────────────────────────────────────────────────
    batch_size       = BATCH_SIZE,
    test_batch_size  = TEST_BATCH_SIZE,
    num_workers      = NUM_WORKERS,

    # ── eval (used by nb_eval_helpers) ──────────────────────────────
    probe_batch_size = TEST_BATCH_SIZE,

    # ── misc ────────────────────────────────────────────────────────
    val_ratio        = float(_hp.get("val_ratio", 0.1)),
    log_interval     = 100,
    dry_run          = False,
    save_model       = True,
    gpu_id           = 0,
    sub_set_mode     = False,
    sub_set_samples  = 10000,
)

print("args namespace ready.")

---
## Cell 5 — Device selection

In [ ]:
if torch.cuda.is_available():
    device = torch.device(f"cuda:{args.gpu_id}")
    torch.cuda.set_device(args.gpu_id)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

---
## Cell 6 — Load dataset (once)

The full dataset is loaded **once** and reused across all `(seed, forget_class)`
iterations.  Retain and forget subsets are assembled per-experiment from the
split JSON files produced by NB1.

In [ ]:
# get_dataset uses args.train_transform and args.dataset / args.data_path
train_dataset_full, test_dataset = get_dataset(args)

# Resolve forget_classes now that num_classes is known
if _forget_classes_cfg is None:
    FORGET_CLASSES: List[int] = list(range(args.num_classes))
else:
    FORGET_CLASSES = [int(c) for c in _forget_classes_cfg]

print(f"Train set size : {len(train_dataset_full)}")
print(f"Test  set size : {len(test_dataset)}")
print(f"num_classes    : {args.num_classes}")
print(f"forget_classes : {FORGET_CLASSES}")

---
## Cell 7 — Helper utilities

Same structural helpers as NB1 plus Oracle-specific path constructors and
dataset-view builders.

**Scheduler note:** when `warmup_epochs = 0`, `SequentialLR` with
`milestones=[0]` is invalid (PyTorch requires milestones > 0 and ≤ total
epochs).  In that case the helper returns a plain `CosineAnnealingLR` instead.

In [ ]:
def set_all_seeds(seed: int) -> None:
    """Pin every RNG source to `seed` for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def oracle_ckpt_path(seed: int, fc: int) -> Path:
    """Canonical Oracle checkpoint path for (seed, forget_class)."""
    return ORACLE_DIR / f"{EXPERIMENT_NAME}_oracle_seed{seed}_fc{fc}{SUFFIX}.pt"


def oracle_log_path(seed: int, fc: int) -> Path:
    return ORACLE_DIR / f"{EXPERIMENT_NAME}_oracle_seed{seed}_fc{fc}{SUFFIX}_trainlog.json"


def split_json_path(seed: int, fc: int) -> Path:
    """Path of the NB1-produced split JSON for (seed, forget_class)."""
    return SPLITS_DIR / f"{EXPERIMENT_NAME}_seed{seed}_fc{fc}{SUFFIX}.json"


def _load_split(seed: int, fc: int) -> Dict[str, Any]:
    """Load and return the split JSON produced by NB1."""
    p = split_json_path(seed, fc)
    assert p.exists(), (
        f"Split file not found: {p}\n"
        "Run NB1 first to generate splits before executing NB2."
    )
    with open(str(p)) as _sf:
        return json.load(_sf)


def _build_subset(base_dataset, idx: List[int]):
    """
    Return a dataset view containing exactly the given indices.
    Supports CIFAR-style (.data / .targets) and ImageFolder-style datasets.
    """
    ds = copy.deepcopy(base_dataset)
    if hasattr(ds, "data"):  # CIFAR-style
        ds.data    = base_dataset.data[idx]
        ds.targets = [base_dataset.targets[i] for i in idx]
    else:                    # ImageFolder-style
        ds.samples = [base_dataset.samples[i] for i in idx]
        if hasattr(ds, "imgs"):
            ds.imgs = ds.samples
        if hasattr(base_dataset, "targets"):
            ds.targets = [base_dataset.targets[i] for i in idx]
        else:
            ds.targets = [s[1] for s in ds.samples]
    return ds


def _split_train_val(dataset, val_ratio: float, seed: int):
    """Deterministic train-val split of an already-filtered dataset."""
    total   = len(dataset)
    val_len = int(total * val_ratio)
    tr_len  = total - val_len
    g = torch.Generator().manual_seed(seed)
    all_idx = torch.randperm(total, generator=g).tolist()
    tr_idx, val_idx = all_idx[:tr_len], all_idx[tr_len:]

    if hasattr(dataset, "data"):  # CIFAR-style
        tr_ds  = copy.deepcopy(dataset)
        val_ds = copy.deepcopy(dataset)
        tr_ds.data  = dataset.data[tr_idx]
        val_ds.data = dataset.data[val_idx]
        tr_ds.targets  = [dataset.targets[i] for i in tr_idx]
        val_ds.targets = [dataset.targets[i] for i in val_idx]
    else:                          # ImageFolder-style
        tr_ds  = copy.deepcopy(dataset)
        val_ds = copy.deepcopy(dataset)
        tr_ds.samples  = [dataset.samples[i] for i in tr_idx]
        val_ds.samples = [dataset.samples[i] for i in val_idx]
        if hasattr(dataset, "imgs"):
            tr_ds.imgs  = tr_ds.samples
            val_ds.imgs = val_ds.samples
        if hasattr(dataset, "targets"):
            tr_ds.targets  = [dataset.targets[i] for i in tr_idx]
            val_ds.targets = [dataset.targets[i] for i in val_idx]
        else:
            tr_ds.targets  = [s[1] for s in tr_ds.samples]
            val_ds.targets = [s[1] for s in val_ds.samples]
    return tr_ds, val_ds


def _make_loader(ds, shuffle: bool, batch_size: int = BATCH_SIZE):
    use_cuda = device.type == "cuda"
    return torch.utils.data.DataLoader(
        ds,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=use_cuda,
        shuffle=shuffle,
    )


def _build_scheduler(optimizer, warmup_ep: int, total_ep: int):
    """Build warmup→cosine scheduler.

    When warmup_ep == 0, SequentialLR with milestones=[0] would raise a
    PyTorch ValueError, so a plain CosineAnnealingLR is used instead.
    When warmup_ep > 0, the schedule is identical to NB1.
    """
    if warmup_ep == 0:
        return CosineAnnealingLR(optimizer, T_max=max(1, total_ep), eta_min=args.min_lr)
    cosine_ep    = max(1, total_ep - warmup_ep)
    warmup_sched = LambdaLR(
        optimizer,
        lr_lambda=lambda cur: min(1.0, (cur + 1) / warmup_ep),
    )
    cosine_sched = CosineAnnealingLR(optimizer, T_max=cosine_ep, eta_min=args.min_lr)
    return SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[warmup_ep],
    )


def _build_model() -> torch.nn.Module:
    """Construct a fresh ModelModule with CMFClassifier=True, remove_FC=True."""
    _args = copy.copy(args)
    _args.CMFClassifier = True
    _args.remove_FC     = True
    model = ModelModule(_args).to(device)
    # Mandatory guard: verify CMFweights attribute is present
    assert hasattr(model, "CMFweights"), (
        "ModelModule did not create CMFweights — check that "
        "CMFClassifier=True and remove_FC=True were accepted."
    )
    return model


print("Helpers defined.")

---
## Cell 8 — Validate NB1 artifact availability and metadata consistency

Before running any training, check that:
1. All required split JSON files exist.
2. Each split file's stored metadata (`experiment_name`, `dataset`, `suffix`)
   matches the current notebook configuration — guards against accidentally
   using splits from a different experiment or a different suffix run.
3. Pretrain checkpoints are present (warning only — not required for Oracle
   training).

Missing or mismatched split files abort the run early with a clear message.

In [ ]:
print("=" * 60)
print("Validating NB1 artifacts ...")
print("=" * 60)

_missing_splits    = []
_mismatch_splits   = []
_missing_pretrain  = []

for seed in SEEDS:
    # ── pretrain checkpoint (metadata reference; Oracle never loads weights) ──
    _pt = PRETRAIN_DIR / f"{EXPERIMENT_NAME}_seed{seed}{SUFFIX}.pt"
    if not _pt.exists():
        _missing_pretrain.append(str(_pt))
        print(f"[WARN]    pretrain checkpoint missing (Oracle can still run): {_pt}")
    else:
        # Light metadata check: load config block from checkpoint without
        # pulling full weights into memory.
        try:
            _pt_state = torch.load(str(_pt), map_location="cpu")
            _pt_cfg   = _pt_state.get("config", {})
            _pt_name  = _pt_cfg.get("experiment_name", "<unknown>")
            _pt_ds    = _pt_cfg.get("dataset",         "<unknown>")
            _match = (_pt_name == EXPERIMENT_NAME and _pt_ds == DATASET)
            _tag = "OK" if _match else "MISMATCH"
            print(f"[{_tag}]   pretrain: {_pt.name}  "
                  f"(experiment='{_pt_name}', dataset='{_pt_ds}')")
        except Exception:
            print(f"[OK]      pretrain checkpoint exists (metadata unreadable): {_pt.name}")

    # ── split files ───────────────────────────────────────────────────────────
    for fc in FORGET_CLASSES:
        _sp = split_json_path(seed, fc)
        if not _sp.exists():
            _missing_splits.append(str(_sp))
            print(f"[MISSING] split: {_sp}")
            continue

        # Metadata consistency check
        try:
            with open(str(_sp)) as _sf:
                _sp_meta = json.load(_sf)
            _sp_name   = _sp_meta.get("experiment_name", "<unknown>")
            _sp_ds     = _sp_meta.get("dataset",         "<unknown>")
            _sp_fc     = _sp_meta.get("forget_class",    -1)
            _sp_suffix = _sp_meta.get("suffix",          "")
            _sp_seed   = _sp_meta.get("seed",            -1)
            _ok = (
                _sp_name   == EXPERIMENT_NAME
                and _sp_ds == DATASET
                and _sp_fc == fc
                and _sp_seed == seed
                and str(_sp_suffix) == str(SUFFIX)
            )
            if _ok:
                print(f"[OK]      split seed={seed} fc={fc}: "
                      f"retain_tr={_sp_meta.get('n_retain_train')}  "
                      f"retain_tst={_sp_meta.get('n_retain_test')}  "
                      f"forget_tst={_sp_meta.get('n_forget_test')}")
            else:
                _mismatch_splits.append((seed, fc, str(_sp)))
                print(f"[MISMATCH] split seed={seed} fc={fc}: "
                      f"stored=(experiment='{_sp_name}', dataset='{_sp_ds}', "
                      f"seed={_sp_seed}, fc={_sp_fc}, suffix='{_sp_suffix}') "
                      f"expected=('{EXPERIMENT_NAME}', '{DATASET}', "
                      f"{seed}, {fc}, '{SUFFIX}')")
        except Exception as _e:
            print(f"[WARN]    split {_sp.name}: metadata unreadable ({_e})")

# ── abort if any split is missing or mismatched ───────────────────────────────
errors = []
if _missing_splits:
    errors.append(
        f"{len(_missing_splits)} split file(s) missing:\n  "
        + "\n  ".join(_missing_splits)
    )
if _mismatch_splits:
    errors.append(
        f"{len(_mismatch_splits)} split file(s) have mismatched metadata:\n  "
        + "\n  ".join(str(t) for t in _mismatch_splits)
    )
if errors:
    raise ValueError(
        "\nNB1 artifact validation failed.  Run NB1 to completion before "
        "executing NB2.\n\n" + "\n\n".join(errors)
    )

if _missing_pretrain:
    print(f"\n[WARN] {len(_missing_pretrain)} pretrain checkpoint(s) missing "
          "(Oracle training will proceed without them).")

print("\nArtifact validation complete — all splits verified.")

---
## Cell 9 — Oracle retraining loop

For each `(seed, forget_class)` pair:
1. Load the NB1 split JSON; extract `retain_train_idx`, `retain_test_idx`,
   and `forget_test_idx`.
2. Build retain-only train dataset and **both** retain and forget test dataset
   views from those indices.
3. Split retain training data into train / val (deterministic 90/10, same as NB1).
4. Initialize a **fresh** model — never warm-started from a pretrained checkpoint.
5. Train with cosine + warmup + early-stopping schedule (same grace-period logic
   as NB1); `warmup_epochs = 0` is handled safely.
6. After training, run the **shared** `eval_cmf_three_metrics` pipeline
   (Output + Linear Probe + NCC) on the correct retain-test and forget-test
   loaders.
7. Save a self-describing Oracle checkpoint that includes all three-metric
   evaluation results.
8. Record a complete row in `oracle_records` for `oracle_summary.csv`.

If a checkpoint already exists the experiment is skipped but the full
three-metric evaluation is **re-run** on the loaded model so the CSV is always
complete.  Any exception is caught; execution continues to the next pair.

In [ ]:
oracle_records: List[Dict[str, Any]] = []   # one row per completed experiment
completed_runs: List[tuple] = []
skipped_runs:   List[tuple] = []
failed_runs:    List[tuple] = []

for seed in SEEDS:
    for fc in FORGET_CLASSES:
        ckpt = oracle_ckpt_path(seed, fc)
        logp = oracle_log_path(seed, fc)

        print(f"\n{'='*60}")
        print(f"Oracle  seed={seed}  forget_class={fc}  |  {ckpt.name}")
        print(f"{'='*60}")

        # ── load split (always needed — for loaders and metadata) ─────────────
        try:
            split_meta = _load_split(seed, fc)
        except AssertionError as _e:
            print(f"[ERROR] {_e}")
            failed_runs.append((seed, fc))
            continue

        retain_tr_idx  = split_meta["retain_train_idx"]
        retain_tst_idx = split_meta["retain_test_idx"]
        forget_tst_idx = split_meta["forget_test_idx"]

        print(f"  retain_train={len(retain_tr_idx)}  "
              f"retain_test={len(retain_tst_idx)}  "
              f"forget_test={len(forget_tst_idx)}")

        # ── build dataset views (needed for skip-path eval too) ───────────────
        retain_train_ds = _build_subset(train_dataset_full, retain_tr_idx)
        # separate test loaders for retain and forget (Issue 1 fix)
        retain_test_loader, forget_test_loader = build_test_split_loaders(
            test_dataset,
            forget_test_indices = forget_tst_idx,
            retain_test_indices = retain_tst_idx,
            batch_size          = TEST_BATCH_SIZE,
            num_workers         = NUM_WORKERS,
        )
        # full test loader for Output metric — original dataset ordering preserved
        full_test_loader = _make_loader(test_dataset, shuffle=False, batch_size=TEST_BATCH_SIZE)

        # ── checkpoint-skip ───────────────────────────────────────────────────
        if ckpt.exists():
            print(f"[SKIP] Oracle checkpoint exists — loading for eval.")
            skipped_runs.append((seed, fc))
            try:
                model = _build_model()
                state = torch.load(str(ckpt), map_location=device)
                if "state_dict" in state:
                    model.load_state_dict(state["state_dict"])
                elif "model_state" in state:
                    model.load_state_dict(state["model_state"])
                else:
                    model.load_state_dict(state)
                model.eval()

                # Re-run full shared evaluation so CSV is always complete
                tr_loader_eval = _make_loader(retain_train_ds, shuffle=False,
                                              batch_size=TEST_BATCH_SIZE)
                eval_metrics = eval_cmf_three_metrics(
                    model, args, device,
                    train_loader       = tr_loader_eval,
                    test_loader        = full_test_loader,
                    retain_test_loader = retain_test_loader,
                    forget_test_loader = forget_test_loader,
                    forget_class       = fc,
                )
                _m = state.get("metrics", {})
                oracle_records.append({
                    "seed":              seed,
                    "forget_class":      fc,
                    "status":            "skipped",
                    "best_epoch":        _m.get("best_epoch"),
                    "best_val_acc":      _m.get("best_val_acc"),
                    "wall_clock_minutes": state.get("wall_clock_minutes"),
                    "n_retain_train":    len(retain_tr_idx),
                    "n_retain_test":     len(retain_tst_idx),
                    "n_forget_test":     len(forget_tst_idx),
                    **{k: round(float(v), 4) if isinstance(v, float) else v
                       for k, v in eval_metrics.items() if not k.startswith("_")},
                    "ckpt_path":         str(ckpt),
                })
            except Exception:
                print(f"[WARN] Could not re-evaluate skipped checkpoint:")
                traceback.print_exc()
            continue

        # ── train Oracle from scratch ─────────────────────────────────────────
        try:
            t0 = time.time()
            set_all_seeds(seed)

            # deterministic 90/10 val split of the retain train subset
            tr_ds, val_ds = _split_train_val(retain_train_ds, args.val_ratio, seed)
            tr_loader  = _make_loader(tr_ds,  shuffle=True)
            val_loader = _make_loader(val_ds, shuffle=False, batch_size=TEST_BATCH_SIZE)

            # fresh model — never warm-started from pretrained checkpoint
            model = _build_model()

            # optimiser
            optimizer = optim.SGD(
                model.parameters(),
                lr=args.lr,
                momentum=args.momentum,
                weight_decay=args.weight_decay,
                nesterov=args.nesterov,
            )

            # ── scheduler: warmup → cosine, safe for warmup_ep=0 ──────────────
            warmup_ep = max(0, args.warmup_epochs)
            total_ep  = args.epochs_or_steps
            scheduler = _build_scheduler(optimizer, warmup_ep, total_ep)

            # ── training state ────────────────────────────────────────────────
            best_val_acc     = 0.0
            best_epoch       = 0
            epochs_no_improv = 0
            history          = {
                "epoch": [], "lr": [],
                "train_loss": [], "train_acc": [],
                "val_retain_acc": [], "val_forget_acc": [], "val_metric": [],
                "test_retain_acc": [], "test_forget_acc": [], "test_metric": [],
            }

            # temporary best-weight file
            _tmp_best = str(ckpt) + ".tmp_best"

            print(f"Training {total_ep} epochs, warmup={warmup_ep}, "
                  f"patience={args.patience}")

            for epoch in range(1, total_ep + 1):
                _tr_args = copy.copy(args)
                _tr_args.seed = seed
                tr_loss, tr_acc = train(
                    _tr_args, model, device, tr_loader, optimizer, epoch, "descent"
                )

                # Validation via shared test() — identical to NB1
                val_r, val_f, val_m = test(
                    model, device, val_loader,
                    [],
                    args.class_label_names, args.num_classes,
                    plot_cm=False, job_name="oracle", verbose=False, set_name="Val"
                )
                tst_r, tst_f, tst_m = test(
                    model, device, full_test_loader,
                    [fc],
                    args.class_label_names, args.num_classes,
                    plot_cm=False, job_name="oracle", verbose=False, set_name="Test"
                )

                cur_lr = float(optimizer.param_groups[0]["lr"])
                history["epoch"].append(epoch)
                history["lr"].append(cur_lr)
                history["train_loss"].append(float(tr_loss))
                history["train_acc"].append(float(tr_acc))
                history["val_retain_acc"].append(float(val_r))
                history["val_forget_acc"].append(float(val_f))
                history["val_metric"].append(val_m)
                history["test_retain_acc"].append(float(tst_r))
                history["test_forget_acc"].append(float(tst_f))
                history["test_metric"].append(tst_m)

                # early-stopping monitors retain validation accuracy
                if val_r > best_val_acc:
                    best_val_acc     = val_r
                    best_epoch       = epoch
                    epochs_no_improv = 0
                    torch.save(model.state_dict(), _tmp_best)
                    print(f"  epoch {epoch:4d} | val_retain {val_r:.4f} ← new best | lr {cur_lr:.6f}")
                else:
                    epochs_no_improv += 1
                    if epoch % 20 == 0:
                        print(f"  epoch {epoch:4d} | val_retain {val_r:.4f} | "
                              f"no-improv {epochs_no_improv}/{args.patience} | lr {cur_lr:.6f}")
                    if epochs_no_improv >= args.patience:
                        print(f"  Early stop at epoch {epoch} "
                              f"(no improvement for {args.patience} epochs).")
                        break

                scheduler.step()

            # reload best weights
            if os.path.exists(_tmp_best):
                model.load_state_dict(torch.load(_tmp_best, map_location=device))
                os.remove(_tmp_best)

            runtime = time.time() - t0

            # ── shared three-metric evaluation (Issue 2 fix) ──────────────────
            # Uses eval_cmf_three_metrics from nb_eval_helpers — same pipeline
            # as NB3 and NB4.  Receives the full retain train set (no-shuffle)
            # to define class centres, plus separate retain-test and forget-test
            # loaders so Output/Probe/NCC are all meaningful (Issue 1 fix).
            model.eval()
            tr_loader_eval = _make_loader(retain_train_ds, shuffle=False,
                                          batch_size=TEST_BATCH_SIZE)
            eval_metrics = eval_cmf_three_metrics(
                model, args, device,
                train_loader       = tr_loader_eval,
                test_loader        = full_test_loader,
                retain_test_loader = retain_test_loader,
                forget_test_loader = forget_test_loader,
                forget_class       = fc,
            )

            print(f"\n  [Eval] Output  retain={eval_metrics.get('output_retain_acc', float('nan')):.4f}  "
                  f"forget={eval_metrics.get('output_forget_acc', float('nan')):.4f}")
            print(f"  [Eval] Probe   retain={eval_metrics.get('probe_retain_acc',  float('nan')):.4f}  "
                  f"forget={eval_metrics.get('probe_forget_acc',  float('nan')):.4f}")
            print(f"  [Eval] NCC     retain={eval_metrics.get('ncc_retain_acc',    float('nan')):.4f}  "
                  f"forget={eval_metrics.get('ncc_forget_acc',    float('nan')):.4f}")
            print(f"  (seed={seed}, fc={fc}, best_epoch={best_epoch})")

            # ── self-describing Oracle checkpoint ─────────────────────────────
            _sd = model.state_dict()
            # eval_metrics may contain a non-serialisable _probe_full dict;
            # store only the scalar metrics in the checkpoint.
            _eval_scalars = {k: to_jsonable(v) for k, v in eval_metrics.items()
                             if not k.startswith("_")}
            checkpoint = {
                "state_dict":     _sd,           # canonical key
                "model_state":    _sd,           # backward-compat alias
                "seed":           seed,
                "forget_class":   fc,
                "stage":          "oracle",
                "config": {
                    "experiment_name": EXPERIMENT_NAME,
                    "suffix":          SUFFIX,
                    "dataset":         DATASET,
                    "arch":            ARCH,
                    "CMFClassifier":   True,
                    "remove_FC":       True,
                    "hp":              {k: to_jsonable(v) for k, v in _hp.items()},
                    "hp_overrides":    _hp_overrides,
                    "split_file":      str(split_json_path(seed, fc)),
                },
                "n_retain_train": len(retain_tr_idx),
                "n_retain_test":  len(retain_tst_idx),
                "n_forget_test":  len(forget_tst_idx),
                "metrics": {
                    "best_epoch":    best_epoch,
                    "best_val_acc":  best_val_acc,
                    **_eval_scalars,
                },
                "wall_clock_minutes": round(runtime / 60, 3),
                "history":        history,
            }
            torch.save(checkpoint, str(ckpt))
            print(f"[SAVED] {ckpt}")

            # training log JSON (human-readable, no tensors)
            log_data = {k: v for k, v in checkpoint.items()
                        if k not in ("state_dict", "model_state")}
            with open(str(logp), "w") as _lf:
                json.dump(log_data, _lf, indent=2, default=to_jsonable)
            print(f"[LOG]   {logp}")

            oracle_records.append({
                "seed":              seed,
                "forget_class":      fc,
                "status":            "completed",
                "best_epoch":        best_epoch,
                "best_val_acc":      round(best_val_acc, 4),
                "wall_clock_minutes": round(runtime / 60, 3),
                "n_retain_train":    len(retain_tr_idx),
                "n_retain_test":     len(retain_tst_idx),
                "n_forget_test":     len(forget_tst_idx),
                **{k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in eval_metrics.items() if not k.startswith("_")},
                "ckpt_path":         str(ckpt),
            })
            completed_runs.append((seed, fc))

        except Exception:  # catch-all so other experiments still run
            print(f"\n[ERROR] Oracle training failed for seed={seed} fc={fc}:")
            traceback.print_exc()
            # clean up dangling tmp file if present
            _tmp = str(ckpt) + ".tmp_best"
            if os.path.exists(_tmp):
                os.remove(_tmp)
            failed_runs.append((seed, fc))

print("\n" + "=" * 60)
print(f"Oracle retraining complete.")
print(f"  Completed : {len(completed_runs)}  {completed_runs}")
print(f"  Skipped   : {len(skipped_runs)}  {skipped_runs}")
print(f"  Failed    : {len(failed_runs)}  {failed_runs}")

---
## Cell 10 — Save oracle_summary.csv

Writes `results_dir/<experiment_name><suffix>_oracle_summary.csv`.
Contains one row per `(seed, forget_class)` experiment with status,
training metadata, and the full three-metric evaluation results
(Output, Linear Probe, NCC) required by NB3–NB4 downstream comparisons.

In [ ]:
if oracle_records:
    summary_df   = pd.DataFrame(oracle_records)
    summary_path = RESULTS_DIR / f"{EXPERIMENT_NAME}{SUFFIX}_oracle_summary.csv"
    summary_df.to_csv(str(summary_path), index=False)
    print(f"Oracle summary saved → {summary_path}")
    display(summary_df)
else:
    print("No Oracle records to summarise (all were skipped or failed).")

---
## Cell 11 — Final output manifest

Print a concise summary of all artefacts produced so the notebook is
self-documenting when run as a report.

In [ ]:
print("="*60)
print("NB2 — OUTPUT MANIFEST")
print("="*60)

print("\n[Oracle checkpoints]")
oracle_files_all = sorted(ORACLE_DIR.glob(f"{EXPERIMENT_NAME}_oracle*{SUFFIX}.pt"))
for p in oracle_files_all:
    size_mb = p.stat().st_size / 1_048_576 if p.exists() else 0
    print(f"  {p}  ({size_mb:.1f} MB)")

print("\n[Oracle training logs]")
log_files_all = sorted(ORACLE_DIR.glob(f"{EXPERIMENT_NAME}_oracle*{SUFFIX}_trainlog.json"))
for p in log_files_all:
    print(f"  {p}")

print("\n[Summary CSV]")
_csv = RESULTS_DIR / f"{EXPERIMENT_NAME}{SUFFIX}_oracle_summary.csv"
print(f"  {_csv}" if _csv.exists() else "  (not produced)")

print(f"\n[Run status]")
print(f"  Completed : {len(completed_runs)}  {completed_runs}")
print(f"  Skipped   : {len(skipped_runs)}  {skipped_runs}")
if failed_runs:
    print(f"\n[WARNING] Oracle training failed for: {failed_runs}")

print("\nVerification checklist:")
print("  [x] Oracle trained from random initialisation (never from pretrain checkpoint)")
print("  [x] Splits loaded from NB1 JSON — never regenerated")
print("  [x] Retain membership from stored split indices only")
print("  [x] Shared eval_cmf_three_metrics pipeline used (Output + Probe + NCC)")
print("  [x] Forget-test loader evaluated separately — all metrics are meaningful")
print("  [x] Oracle checkpoints compatible with downstream NB3–NB4")
print("  [x] oracle_summary.csv contains complete three-metric evaluation")
print("\nNB2 complete. Downstream notebooks should consume the Oracle checkpoints above.")